In [155]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import faiss
import json
from pathlib import Path
from ollama import embed,chat

In [2]:
### loading the dataset 

ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled")

c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\datasets--qiaojin--PubMedQA. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 1000/1000 [00:00<00:00, 19219.74 examples/s]


In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})

In [7]:
df = pd.DataFrame(ds['train'])  ### creating dataframe from ds

In [8]:
df.head()

,pubid,question,context,long_answer,final_decision
0,21645374,Do mitochondria play a role in remodelling lac...,{'contexts': ['Programmed cell death (PCD) is ...,Results depicted mitochondrial dynamics in viv...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,{'contexts': ['Assessment of visual acuity dep...,"Using the charts described, there was only a s...",no
2,9488747,"Syncope during bathing in infants, a pediatric...",{'contexts': ['Apparent life-threatening event...,"""Aquagenic maladies"" could be a pediatric form...",yes
3,17208539,Are the long-term results of the transanal pul...,{'contexts': ['The transanal endorectal pull-t...,Our long-term study showed significantly bette...,no
4,10808977,Can tailored interventions increase mammograph...,{'contexts': ['Telephone counseling and tailor...,The effects of the intervention were most pron...,yes


In [122]:
def get_context_in_list(df:pd.DataFrame) -> list:
    context_list = []
    for _,row in df.iterrows():
        for cntxt in row['context']['contexts']:
            if len(cntxt.split()) > 20:
                context_list.append(cntxt)

    return context_list


In [127]:
context_list = get_context_in_list(df) ## saving all context in a list

In [128]:
len(context_list)

2886

In [129]:
### getting embeddings for all context 

context_embeddings = embed(
    model='embeddinggemma',
    input=context_list,
)

In [130]:
context_embeddings = np.array(context_embeddings['embeddings'],dtype= np.float32)

In [131]:
context_embeddings.shape

(2886, 768)

In [132]:
faiss.normalize_L2(context_embeddings)

In [133]:
dim = context_embeddings.shape[1] ### define dim for FAISS
index = faiss.IndexHNSWFlat(dim,32,faiss.METRIC_INNER_PRODUCT) ## initiating the FAISS

index.hnsw.efConstruction = 200
index.hnsw.efSearch = 50


In [134]:
index.add(context_embeddings)

In [135]:
index.ntotal  ### total number of embeddings saved now

2886

In [136]:
cwd = Path.cwd()
cwd

WindowsPath('c:/Users/ADMIN/MLops/ollama/notebook')

In [137]:
#### saved index as a file 

index_path = cwd.parent.as_posix() + '/data/processed/medical_rag.index'
faiss.write_index(index,index_path)

In [138]:
load_index = faiss.read_index(index_path)  ### load index 

In [139]:
query = "what causes cell death?" ## query from user
q_emb = embed(model='embeddinggemma',
    input=[query])['embeddings']
q_emb = np.array(q_emb,dtype=np.float32)

In [140]:
faiss.normalize_L2(q_emb)

In [141]:
scores,indices = load_index.search(q_emb,3)

In [142]:
scores[0]

array([0.44036964, 0.3843407 , 0.38365012], dtype=float32)

In [143]:
indices

array([[  0, 219, 133]])

In [ ]:
THRESHOLD = 0.4

results = []
for score, idx in zip(scores[0], indices[0]):
    if score >= THRESHOLD:
        results.append({
            "chunk": context_list[idx],   # ← look up text
            "score": float(score)
        })
    
for r in results:
    print(f"{r['score']:.3f} → {r['chunk']}")

0.440 → Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.


In [147]:
retrieved_context = []
for idx in results:
    retrieved_context.append(idx['chunk'])

retrieved_context = ''.join(retrieved_context)

In [148]:
retrieved_context

'Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.'

In [149]:
prompt = f'''
            You are a clinical doctor and have good exertise in medicine and surgery anmd having excellent experinec from top to bottom about a disease
            Answer the query using ONLY the context below.
            Return the response in a JSON format.

            ## Context:
            {retrieved_context}

            ## Question:
            {query}

            ## Answer
            ""
'''

prompt.format(retrieved_context = retrieved_context,query=query
              )

'\n            You are a clinical doctor and have good exertise in medicine and surgery anmd having excellent experinec from top to bottom about a disease\n            Answer the query using ONLY the context below.\n            Return the response in a JSON format.\n\n            ## Context:\n            Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.\n\n            ## Question:\n            what causes cell death?\n\n            ## Answer\n            ""\n'

In [153]:
stream = chat(
    model='gemma4:latest',
    messages=[{'role': 'user', 'content':prompt}],
    stream=False,
    think = False

    )
# for chunk in stream:
#     print(chunk['message']['content'], end='', flush=True)

In [162]:
import re

match = re.search(
            r'```json\s*(.*?)\s*```',
            stream['message']['content'],
            re.DOTALL
        )
if match:
    parsed = json.loads(match.group(1))
    print(parsed)

{'answer': 'Programmed cell death (PCD) is the regulated death of cells within an organism.'}


In [157]:
stream['message']['content']

'```json\n{\n  "answer": "Programmed cell death (PCD) is the regulated death of cells within an organism."\n}\n```'